In [17]:
import pandas as pd

# Datasets
iea_p = pd.read_csv("/Users/cynthiasalinas/Documents/econ_climate/iea_p.csv", encoding="latin1")
iea_policies_raw = pd.read_csv("/Users/cynthiasalinas/Documents/econ_climate/IEA_policies_raw.csv", encoding="latin1")

# Merge by country and year
merged_df = iea_p.merge(iea_policies_raw[['country', 'year', 'status']],
                         on=['country', 'year'], how='left')

# New DataFrame
merged_df.to_csv("/Users/cynthiasalinas/Documents/econ_climate/iea_p_merged.csv", index=False)

merged_df.head()

,year,country,iso3,pol,status
0,2011,Morocco,MAR,2,In force
1,2011,Morocco,MAR,2,In force
2,2022,Switzerland,CHE,6,Announced
3,2022,Switzerland,CHE,6,In force
4,2022,Switzerland,CHE,6,In force


In [18]:
status_counts = merged_df["status"].value_counts()
print(status_counts)

status
In force     8044
Ended        1586
Announced     325
Achieved        4
Planned         1
Name: count, dtype: int64


In [24]:
df_filtered = merged_df[merged_df["status"] == "In force"].copy()
print(df_filtered.head())
print(" ")

df_filtered_afgh = df_filtered[df_filtered["country"] == "Afghanistan"]
print(df_filtered_afgh.head())

   year      country iso3  pol    status
0  2011      Morocco  MAR    2  In force
1  2011      Morocco  MAR    2  In force
3  2022  Switzerland  CHE    6  In force
4  2022  Switzerland  CHE    6  In force
5  2022  Switzerland  CHE    6  In force
 
      year      country iso3  pol    status
2285  2021  Afghanistan  AFG    1  In force
8139  2019  Afghanistan  AFG    1  In force
8953  2006  Afghanistan  AFG    1  In force


In [32]:
df_filtered = merged_df[merged_df["status"] == "In force"].copy()
# Policies from 2000 to 2022, where a record is shown per year
expanded_rows = []
for _, row in df_filtered.iterrows():
    for year in range(row["year"], 2023):  # Hasta 2022 incluido
        expanded_rows.append({
            "country": row["country"],
            "iso3": row["iso3"],
            "year": year
        })

# New DataFrame
df_final = pd.DataFrame(expanded_rows)

# Final DataFrame: Where policies are shown per active year + a new column "pol_count" where the number adds up if there are more than one policy active per year"
df_final["pol_count"] = df_final.groupby(["country", "year"]).cumcount() + 1
df_final.to_csv("/Users/cynthiasalinas/Documents/econ_climate/iea_pol_per_year.csv", index=False)
print(df_final.head())
print(" ")

#Afghanistan Example for visualization on the changes made
df_final_afgh = df_final[df_final["country"] == "Afghanistan"].sort_values(by="year", ascending=False)
print(df_final_afgh.head(10))

   country iso3  year  pol_count
0  Morocco  MAR  2011          1
1  Morocco  MAR  2012          1
2  Morocco  MAR  2013          1
3  Morocco  MAR  2014          1
4  Morocco  MAR  2015          1
 
           country iso3  year  pol_count
59853  Afghanistan  AFG  2022          3
52657  Afghanistan  AFG  2022          2
13700  Afghanistan  AFG  2022          1
59852  Afghanistan  AFG  2021          3
13699  Afghanistan  AFG  2021          1
52656  Afghanistan  AFG  2021          2
59851  Afghanistan  AFG  2020          2
52655  Afghanistan  AFG  2020          1
52654  Afghanistan  AFG  2019          1
59850  Afghanistan  AFG  2019          2
